In [ ]:
import argparse
import json
import os
import csv
import torch
from tqdm import tqdm
from PIL import Image
import os
import csv
import torch
from tqdm import tqdm
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# 1. 경로 및 설정 변수 (기본값 설정)
MODEL_PATH = "Lin-Chen/ShareGPT4V-7B"
MODEL_NAME = "share4v-7b"
IMAGE_FOLDER = 'runtime/Camera'
OUTPUT_CSV = "results_plusdata.csv"

# CUDA 라이브러리 경로 강제 추가
os.environ['PATH'] = 'runtime/bin' + os.pathsep + os.environ['PATH']
#os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from share4v.constants import (DEFAULT_IM_END_TOKEN, DEFAULT_IM_START_TOKEN,
                               DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX)
from share4v.conversation import (SeparatorStyle, conv_templates)
from share4v.mm_utils import (process_images, tokenizer_image_token, KeywordsStoppingCriteria)
from share4v.model.builder import load_pretrained_model

# 장치 설정
torch.backends.cuda.matmul.allow_tf32 = True
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"Using Device: {device}")

def load_image(image_file):
    return Image.open(image_file).convert('RGB')

@torch.inference_mode()
def generate_caption(model, tokenizer, image_processor, image_path, prompt_text):
    if image_processor is None:
        vision_tower = model.get_vision_tower()
        if not vision_tower.is_loaded:
            vision_tower.load_model()
        vision_tower.to(device='cuda', dtype=torch.float16)
        image_processor = vision_tower.image_processor

    conv = conv_templates["vicuna_v1"].copy()
    qs = prompt_text
    if getattr(model.config, 'mm_use_im_start_end', False):
        qs = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + '\n' + qs
    else:
        qs = DEFAULT_IMAGE_TOKEN + '\n' + qs

    conv.append_message(conv.roles[0], qs)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()

    image_tensor = process_images([load_image(image_path)], image_processor, model.config)
    image_tensor = image_tensor.to(device, dtype=dtype)

    input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).to(device)
    
    stop_str = conv.sep if conv.sep_style == SeparatorStyle.SINGLE else conv.sep2
    stopping_criteria = KeywordsStoppingCriteria([stop_str], tokenizer, input_ids)

    output_ids = model.generate(
        input_ids,
        images=image_tensor,
        do_sample=True,
        temperature=0.2,
        max_new_tokens=512,
        use_cache=False,
        stopping_criteria=[stopping_criteria]
    )

    outputs = tokenizer.batch_decode(output_ids[:, input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
    if outputs.endswith(stop_str):
        outputs = outputs[:-len(stop_str)]
    return outputs.strip()

def get_image_metadata(image_path):
    """이미지에서 시간, 장소 정보를 추출하고 상태 코드를 반환합니다."""
    img = Image.open(image_path)
    exif_data = img._getexif()
    
    dt = None
    lat = None
    lon = None
    status = 0 # 기본값: 둘 다 없음
    
    if exif_data:
        # 1. 날짜 추출
        for tag, value in exif_data.items():
            tag_name = TAGS.get(tag, tag)
            if tag_name == 'DateTimeOriginal' or tag_name == 'DateTime':
                dt = value
                break
        
        # 2. 장소(GPS) 추출
        gps_info = {}
        for tag, value in exif_data.items():
            if TAGS.get(tag) == 'GPSInfo':
                for t in value:
                    gps_info[GPSTAGS.get(t, t)] = value[t]
                break
        
        if gps_info.get('GPSLatitude') and gps_info.get('GPSLongitude'):
            # GPS 데이터가 존재하면 위도/경도 문자열로 저장
            lat = gps_info.get('GPSLatitude')
            lon = gps_info.get('GPSLongitude')
            location = f"{lat}, {lon}"
        else:
            location = None

        # 3. 상태 코드 결정
        if dt and location:
            status = 2 # 둘 다 있음
        elif dt:
            status = 1 # 시간만 있음
        else:
            status = 0 # 둘 다 없음 (혹은 장소만 있는 특이 케이스도 0 처리)
            
    return dt, location if 'location' in locals() else None, status

def main():
    print(f"Loading model: {MODEL_NAME} to {device}...")
    
    # 모델 로드 (변수로 직접 전달)
    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path=MODEL_PATH,
        model_base=None,
        model_name=MODEL_NAME,
        load_8bit=False,
        load_4bit=False,
        device_map='auto',
        #device="cuda",
    )

    if device == "cpu":
        model = model.float()

    # 이미지 파일 목록 확보
    image_exts = ('.png', '.jpg', '.jpeg', '.webp', '.bmp')
    image_files = [f for f in os.listdir(IMAGE_FOLDER) if f.lower().endswith(image_exts)]

    results = []
    for filename in tqdm(image_files, desc="Processing"):
        image_path = os.path.join(IMAGE_FOLDER, filename)
        full_path = os.path.abspath(image_path) # 이미지 절대 경로
        
        # 메타데이터 추출
        try:
            date_time, location, status = get_image_metadata(image_path)
        except Exception:
            date_time, location, status = None, None, 0

        # AI 캡션 생성
        try:
            description = generate_caption(model, tokenizer, image_processor, image_path, "explain this picture")
        except Exception as e:
            description = f"Error: {str(e)}"

        # 결과 리스트 구성
        # [파일명, 설명, 촬영시간, 장소메타, 이미지경로, 상태코드]
        results.append([filename, description, date_time, location, full_path, status])

    # 결과 저장
    with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        # 헤더 추가
        writer.writerow(['File Name', 'Description', 'DateTime', 'Location', 'Full Path', 'Status'])
        writer.writerows(results)
        
        print(f"Done! Saved to {OUTPUT_CSV}")
if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import json
import os

# 1. ShareGPT4V 데이터 로드 (Hugging Face)
print("ShareGPT4V 데이터를 로드 중입니다...")
df_sharegpt = pd.read_json("hf://datasets/Lin-Chen/ShareGPT4V/sharegpt4v_instruct_gpt4-vision_cap100k.json")

# 2. COCO 2017 메타데이터 로드 (Local JSON)
json_path = 'runtime/captions_train2017.json'
print("COCO 메타데이터를 로드 중입니다...")
with open(json_path, "r", encoding="utf-8") as f:
    coco_data = json.load(f)

# COCO 이미지 메타데이터(id, file_name, date_captured)를 DataFrame으로 변환
# coco_data['images'] 리스트 안에 file_name과 date_captured가 포함되어 있습니다.
df_coco_meta = pd.DataFrame(coco_data['images'])[['file_name', 'date_captured']]

# 3. 데이터 가공 및 통합 준비
def extract_info(row):
    # ShareGPT4V의 'image' 컬럼에서 파일명만 추출 (예: coco/train2017/000000.jpg -> 000000.jpg)
    file_name = os.path.basename(row['image'])
    
    # 캡션 추출: conversations[1]['value'] (GPT의 답변 부분)
    # 데이터 구조에 따라 인덱스 체크가 필요할 수 있으나, 요청하신 대로 [1]['value'] 사용
    try:
        description = row['conversations'][1]['value']
    except (IndexError, KeyError):
        description = ""
        
    # 전체 경로 생성
    full_path = 'runtime/coco2017' + row['image']
    
    return pd.Series([file_name, description, full_path])

print("ShareGPT4V 데이터를 가공 중입니다...")
df_sharegpt[['File Name', 'Description', 'Full Path']] = df_sharegpt.apply(extract_info, axis=1)

# 4. COCO 메타데이터와 병합 (File Name 기준)
print("데이터를 병합 중입니다...")
final_df = pd.merge(df_sharegpt, df_coco_meta, left_on='File Name', right_on='file_name', how='left')

# 5. 최종 컬럼 구성 및 정리
# 요청 형식: File Name, Description, DateTime, Location, Full Path, Status
final_df['DateTime'] = final_df['date_captured']
final_df['Location'] = ""  # 비워둠
final_df['Status'] = 1     # 1로 설정

# 필요한 컬럼만 선택
result_csv = final_df[['File Name', 'Description', 'DateTime', 'Location', 'Full Path', 'Status']]

# 6. CSV 파일로 저장
output_path = "coco_sharegpt_integrated.csv"
result_csv.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"작업 완료! 파일이 저장되었습니다: {output_path}")
print(result_csv.head())

In [ ]:
import torch
import pandas as pd
import os
import gc
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

# 1. 환경 변수 설정
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 2. 모델 및 양자화 설정 (임베딩 추출을 위해 .pt 전용 폴더로 변경)
model_id = "Qwen/Qwen3.5-9B"
save_dir = 'runtime/embedding_results_448_pt_images_only'
os.makedirs(save_dir, exist_ok=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("모델 로딩 중...")
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

# 3. CSV 로드
csv_path = 'runtime/results.csv'
#csv_path = 'runtime/coco_sharegpt_integrated.csv'
df = pd.read_csv(csv_path)

# 4. 루프 실행
for _, row in tqdm(df.iterrows(), total=len(df), desc="고효율 .pt 추출"):
    file_name = row['File Name']
    #description = row['Description']
    full_path = row['Full Path']
    # 확장자를 .pt로 변경
    pt_file_path = os.path.join(save_dir, f"{file_name}.pt")
    
    if os.path.exists(pt_file_path): continue

    try:
        with Image.open(full_path) as img:
            image = img.convert("RGB")
            image.thumbnail((448, 448)) 
        
        messages = [{"role": "user", "content": [{"type": "image"}]}]
        prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
        
        inputs = processor(text=[prompt], images=[image], return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            
        embedding_dict = {}
        
        # [핵심 수정] 텐서를 리스트화(.tolist())하지 않고 텐서 객체 그대로 저장합니다.
        for i, layer_tensor in enumerate(outputs.hidden_states):
            # 1. 배치 차원 제거 [Sequence, 4096]
            # 2. 용량 절감을 위해 float16으로 변환
            # 3. GPU 메모리 확보를 위해 CPU로 즉시 오프로드
            if i==9 or i==19 or i==24:
                embedding_dict[f"hidden_states{i}"] = layer_tensor.squeeze(0).to(torch.float16).cpu()
            
        # [핵심 수정] torch.save를 사용하여 이진(Binary) 파일로 압축 저장
        torch.save(embedding_dict, pt_file_path)

        # 메모리 강제 해제
        del outputs
        del inputs
        del image
        del embedding_dict
        
        # 50회마다 캐시 비우기 (i는 tqdm의 인덱스를 받기 위해 enumerate 사용 권장이나 기존 구조 유지)
        # 루프 횟수 추적을 위해 간단한 카운터 변수 활용 가능
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        tqdm.write(f"에러 발생 ({file_name}): {e}")
        gc.collect()
        torch.cuda.empty_cache()

print(f"\n✨ 작업 완료! 저장 폴더: {save_dir}")

In [ ]:
import torch
import pandas as pd
import os
import gc
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

# 1. 환경 변수 설정
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 2. 모델 및 양자화 설정 (임베딩 추출을 위해 .pt 전용 폴더로 변경)
model_id = "Qwen/Qwen3.5-9B"
save_dir = 'runtime/embedding_results_448_pt_coco'
os.makedirs(save_dir, exist_ok=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("모델 로딩 중...")
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

# 3. CSV 로드
#csv_path = 'runtime/results.csv'
csv_path = 'runtime/coco_sharegpt_integrated.csv'
df = pd.read_csv(csv_path)

# 4. 루프 실행
for _, row in tqdm(df.iterrows(), total=len(df), desc="고효율 .pt 추출"):
    file_name = row['File Name']
    description = row['Description']
    full_path = row['Full Path']
    # 확장자를 .pt로 변경
    pt_file_path = os.path.join(save_dir, f"{file_name}.pt")
    
    if os.path.exists(pt_file_path): continue

    try:
        with Image.open(full_path) as img:
            image = img.convert("RGB")
            image.thumbnail((448, 448)) 
        
        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": str(description)}]}]
        prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
        
        inputs = processor(text=[prompt], images=[image], return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            
        embedding_dict = {}
        
        # [핵심 수정] 텐서를 리스트화(.tolist())하지 않고 텐서 객체 그대로 저장합니다.
        for i, layer_tensor in enumerate(outputs.hidden_states):
            # 1. 배치 차원 제거 [Sequence, 4096]
            # 2. 용량 절감을 위해 float16으로 변환
            # 3. GPU 메모리 확보를 위해 CPU로 즉시 오프로드
            if i==9 or i==19 or i==24:
                embedding_dict[f"hidden_states{i}"] = layer_tensor.squeeze(0).to(torch.float16).cpu()
            
        # [핵심 수정] torch.save를 사용하여 이진(Binary) 파일로 압축 저장
        torch.save(embedding_dict, pt_file_path)

        # 메모리 강제 해제
        del outputs
        del inputs
        del image
        del embedding_dict
        
        # 50회마다 캐시 비우기 (i는 tqdm의 인덱스를 받기 위해 enumerate 사용 권장이나 기존 구조 유지)
        # 루프 횟수 추적을 위해 간단한 카운터 변수 활용 가능
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        tqdm.write(f"에러 발생 ({file_name}): {e}")
        gc.collect()
        torch.cuda.empty_cache()

print(f"\n✨ 작업 완료! 저장 폴더: {save_dir}")

In [ ]:
import torch
import os

file_path = 'runtime/000000000009.jpg.pt'

if os.path.exists(file_path):
    # 데이터 로드
    data = torch.load(file_path)
    
    # 용량 확인
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"📊 파일명: {os.path.basename(file_path)}")
    print(f"⚖️ 파일 용량: {file_size_mb:.2f} MB")
    
    # 첫 번째 레이어의 텐서 구조 확인
    sample_tensor = data['hidden_states9'] # 골든 레이어 중 하나 확인
    seq_len, hidden_dim = sample_tensor.shape
    
    print("\n--- [데이터 규격 검증] ---")
    print(f"📏 시퀀스 길이 (L): {seq_len}")
    print(f"💎 임베딩 차원 (D): {hidden_dim} (Qwen 3.5-9B의 4096 차원)")
    print(f"🧬 데이터 타입: {sample_tensor.dtype}")
    
    # 비판적 분석
    print("\n--- [공학적 분석] ---")
    if hidden_dim == 4096:
        print("✅ 차원 일치: Qwen 3.5-9B의 원본 정보를 손실 없이 보존하고 있습니다.")
    
    if seq_len > 500:
        print(f"⚠️ 경고: 시퀀스가 {seq_len}으로 매우 깁니다. Q-Former 설계 시 이 길이를 고려한 메모리 할당이 필요합니다.")
    else:
        print(f"✅ 안정권: 최대 시퀀스 {seq_len}은 RTX 5080 환경에서 충분히 병렬 처리가 가능한 크기입니다.")

In [ ]:
import math

import torch
import torch.nn as nn

from .rff import GaussianEncoding


# Constants
MINUTE_TO_HOUR  = 1 / 60
SECOND_TO_HOUR  = MINUTE_TO_HOUR / 60

DAY_TO_MONTH    = 1 / (365 / 12)
HOUR_TO_MONTH   = DAY_TO_MONTH / 24
MINUTE_TO_MONTH = MINUTE_TO_HOUR * HOUR_TO_MONTH
SECOND_TO_MONTH = SECOND_TO_HOUR * HOUR_TO_MONTH


def angular_time_representation(T):
    month, day, hour, minute, second = torch.chunk(T.float(), 5, dim=1)

    # Convert integer local datetime to month and day with decimal places
    month_d = (month - 1) + (day - 1) * DAY_TO_MONTH
    hour_d = hour + minute * MINUTE_TO_HOUR + second * SECOND_TO_HOUR

    # Transform month and day to angles between [-pi, +pi)
    theta = 2 * math.pi * month_d / 12 - math.pi
    phi = 2 * math.pi * hour_d / 24 - math.pi

    return torch.cat((theta, phi), dim=1)


class TimeEncoderCapsule(nn.Module):
    def __init__(self, sigma, embedding_dim=512, dropout_prob=None):
        super(TimeEncoderCapsule, self).__init__()
        rff_encoding = GaussianEncoding(sigma=sigma, input_size=2, encoded_size=embedding_dim//2)
        self.sigma = sigma
        self.capsule = nn.Sequential(rff_encoding,
                                     nn.Linear(embedding_dim, 1024),
                                     nn.ReLU(),
                                     nn.Dropout(dropout_prob) if dropout_prob else nn.Identity(),
                                     nn.Linear(1024, 1024),
                                     nn.ReLU(),
                                     nn.Dropout(dropout_prob) if dropout_prob else nn.Identity(),
                                     nn.Linear(1024, 1024),
                                     nn.ReLU(),
                                     nn.Dropout(dropout_prob) if dropout_prob else nn.Identity())
        self.head = nn.Sequential(nn.Linear(1024, embedding_dim))

    def forward(self, x):
        x = self.capsule(x)
        x = self.head(x)
        return x


class TimeEncoder(nn.Module):
    def __init__(self, sigma=[2**0, 2**1, 2**2, 2**3,2**4,2**5,2**6,2**7, 2**8,2**9], embedding_dim=512, dropout_prob=None):
        super(TimeEncoder, self).__init__()
        self.sigma = sigma
        self.n = len(self.sigma)
        self.embedding_dim = embedding_dim

        for i, s in enumerate(self.sigma):
            self.add_module('TimeEnc' + str(i), TimeEncoderCapsule(sigma=s, embedding_dim=embedding_dim, dropout_prob=dropout_prob))

    def forward(self, time):
        time = angular_time_representation(time)
        time_features = torch.zeros(time.shape[0], self.embedding_dim, device=time.device)

        for i in range(self.n):
            time_features += self._modules['TimeEnc' + str(i)](time)
        
        return time_features

In [ ]:
import os
import json
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from datetime import datetime
import math
from tqdm import tqdm

# --- [0. 결과 저장 설정] ---
RESULT_SAVE_DIR = "analysis_result_sequence"
os.makedirs(RESULT_SAVE_DIR, exist_ok=True)

# --- [1. 시간 표현형 및 손실 함수] ---
DAY_TO_MONTH = 1 / (365 / 12)
MINUTE_TO_HOUR = 1 / 60
SECOND_TO_HOUR = MINUTE_TO_HOUR / 60

def angular_time_representation(T):
    month, day, hour, minute, second = torch.chunk(T.float(), 5, dim=1)
    month_d = (month - 1) + (day - 1) * DAY_TO_MONTH
    hour_d = hour + minute * MINUTE_TO_HOUR + second * SECOND_TO_HOUR
    theta = 2 * math.pi * month_d / 12 - math.pi
    phi = 2 * math.pi * hour_d / 24 - math.pi
    return torch.cat((theta, phi), dim=1)

def cyclic_loss(pred, target):
    loss_theta = 1 - torch.cos(pred[:, 0] - target[:, 0])
    loss_phi = 1 - torch.cos(pred[:, 1] - target[:, 1])
    return (loss_theta + loss_phi).mean()

# --- [2. 데이터셋: 상세 로드 상태 보고] ---
class FullSequenceDataset(Dataset):
    def __init__(self, pt_dir, csv_path, layer_idx):
        self.pt_dir = pt_dir
        self.layer_key = f"hidden_states{layer_idx}"
        
        df = pd.read_csv(csv_path)
        df = df[df['Status'].astype(str) != '0'].copy()
        
        self.data_list = []
        skipped_no_file = 0
        skipped_parse_error = 0

        for _, row in df.iterrows():
            img_name = str(row['File Name']).strip()
            pt_path = os.path.join(pt_dir, f"{img_name}.pt")
            
            if os.path.exists(pt_path):
                try:
                    dt_str = str(row['DateTime']).strip()
                    dt = datetime.strptime(dt_str, '%Y:%m:%d %H:%M:%S')
                    time_vec = [dt.month, dt.day, dt.hour, dt.minute, dt.second]
                    self.data_list.append((pt_path, time_vec))
                except:
                    skipped_parse_error += 1
                    continue
            else:
                skipped_no_file += 1
        
        # 첫 번째 레이어 탐색 시에만 요약 보고
        if layer_idx == 0:
            print(f"\n--- [데이터 로딩 요약] ---")
            print(f"✅ 유효 데이터: {len(self.data_list)}개")
            print(f"❌ 파일 없음: {skipped_no_file}개")
            print(f"❌ 날짜 파싱 실패: {skipped_parse_error}개")
            print(f"------------------------\n")

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        pt_path, time_vec = self.data_list[idx]
        data = torch.load(pt_path, map_location='cpu')
        hidden_state = data[self.layer_key].to(torch.float32)
        return hidden_state, torch.tensor(time_vec, dtype=torch.float32)

def collate_fn(batch):
    embeddings, times = zip(*batch)
    padded_embeddings = pad_sequence(embeddings, batch_first=True)
    lengths = torch.tensor([e.size(0) for e in embeddings])
    mask = torch.arange(padded_embeddings.size(1))[None, :] < lengths[:, None]
    return padded_embeddings, torch.stack(times), mask

# --- [3. Attention Pooling 모델] ---
class SequenceProbeModel(nn.Module):
    def __init__(self, input_dim=4096):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.Tanh(),
            nn.Linear(512, 1)
        )
        self.head = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 2)
        )

    def forward(self, x, mask):
        attn_scores = self.attention(x).squeeze(-1)
        attn_scores = attn_scores.masked_fill(~mask, -1e9)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        context_vec = torch.sum(attn_weights * x, dim=1)
        return self.head(context_vec)

# --- [4. 실행 루프 및 리포팅] ---
def run_정밀_탐색(pt_dir, csv_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🖥️ 642 시퀀스 정밀 탐색 시작 | 디바이스: {device}")
    
    layer_results = {}

    for layer_idx in range(33):
        dataset = FullSequenceDataset(pt_dir, csv_path, layer_idx)
        if len(dataset) < 10:
            print(f"⚠️ Layer {layer_idx:02d}: 데이터 부족으로 건너뜁니다.")
            continue

        train_size = int(0.8 * len(dataset))
        val_size = len(dataset) - train_size
        train_ds, val_ds = random_split(dataset, [train_size, val_size])
        
        train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_ds, batch_size=8, collate_fn=collate_fn)

        model = SequenceProbeModel().to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-4)

        best_val_loss = float('inf')
        # 에포크 진행 상황을 postfix로 실시간 모니터링
        epoch_pbar = tqdm(range(10), desc=f"🔍 Layer {layer_idx:02d}", leave=False)
        
        for epoch in epoch_pbar:
            model.train()
            train_loss = 0
            for h_state, t_raw, mask in train_loader:
                h_state, t_raw, mask = h_state.to(device), t_raw.to(device), mask.to(device)
                target = angular_time_representation(t_raw)
                
                optimizer.zero_grad()
                loss = cyclic_loss(model(h_state, mask), target)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            model.eval()
            val_loss = 0
            with torch.no_grad():
                for h_state, t_raw, mask in val_loader:
                    h_state, t_raw, mask = h_state.to(device), t_raw.to(device), mask.to(device)
                    val_loss += cyclic_loss(model(h_state, mask), angular_time_representation(t_raw)).item()
            
            avg_val_loss = val_loss / len(val_loader)
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
            
            epoch_pbar.set_postfix({"Best": f"{best_val_loss:.4f}", "Curr": f"{avg_val_loss:.4f}"})

        print(f"📍 Layer {layer_idx:02d} | Best Cyclic Loss: {best_val_loss:.6f}")
        layer_results[layer_idx] = best_val_loss

    # --- [5. 최종 결과 저장 및 랭킹 보고] ---
    ranking = sorted(layer_results.items(), key=lambda x: x[1])
    
    # 5-1. JSON 상세 리포트
    with open(os.path.join(RESULT_SAVE_DIR, "full_sequence_report.json"), "w") as f:
        json.dump(layer_results, f, indent=4)
    
    # 5-2. TXT 랭킹 요약
    txt_path = os.path.join(RESULT_SAVE_DIR, "final_ranking_summary.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("🏆 [642 시퀀스 기반 레이어 정보량 랭킹]\n")
        f.write("="*50 + "\n")
        for i, (l_idx, loss) in enumerate(ranking):
            line = f"Rank {i+1:02d}: Layer {l_idx:02d} (Loss: {loss:.6f})\n"
            f.write(line)
            if i < 5: print(f"✨ {line.strip()}")
        f.write("="*50 + "\n")

    print(f"\n✅ 분석 완료! 모든 결과는 '{RESULT_SAVE_DIR}' 폴더에 저장되었습니다.")

if __name__ == "__main__":
    # 홍철 님의 실제 경로로 설정
    PT_DIR = 'runtime/embedding_results_448_pt'
    CSV_PATH = 'runtime/results.csv'
    
    run_정밀_탐색(PT_DIR, CSV_PATH)

In [ ]:
import os
import json
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from datetime import datetime, timedelta
import math
import numpy as np
from tqdm import tqdm

# --- [0. 결과 저장 설정] ---
RESULT_SAVE_DIR = "layer11_analysis_result"
os.makedirs(RESULT_SAVE_DIR, exist_ok=True)

# --- [1. 시간 표현형 및 역변환 함수] ---
DAY_TO_MONTH = 1 / (365 / 12)
MINUTE_TO_HOUR = 1 / 60
SECOND_TO_HOUR = MINUTE_TO_HOUR / 60

def angular_time_representation(T):
    month, day, hour, minute, second = torch.chunk(T.float(), 5, dim=1)
    month_d = (month - 1) + (day - 1) * DAY_TO_MONTH
    hour_d = hour + minute * MINUTE_TO_HOUR + second * SECOND_TO_HOUR
    theta = 2 * math.pi * month_d / 12 - math.pi
    phi = 2 * math.pi * hour_d / 24 - math.pi
    return torch.cat((theta, phi), dim=1)

def inverse_angular_time(theta, phi):
    """예측된 각도를 다시 시간으로 (2026년 기준)"""
    month_d = ((theta + math.pi) / (2 * math.pi)) * 12
    month = int(month_d) + 1
    day_fraction = (month_d - int(month_d)) / DAY_TO_MONTH
    day = int(day_fraction) + 1
    hour_d = ((phi + math.pi) / (2 * math.pi)) * 24
    hour = int(hour_d) % 24
    minute_d = (hour_d - int(hour_d)) * 60
    minute = int(minute_d) % 60
    second = int((minute_d - int(minute_d)) * 60) % 60
    
    # 예외 범위 보정
    month = max(1, min(12, month))
    day = max(1, min(28, day))
    return datetime(2026, month, day, hour, minute, second)

def cyclic_loss(pred, target):
    loss_theta = 1 - torch.cos(pred[:, 0] - target[:, 0])
    loss_phi = 1 - torch.cos(pred[:, 1] - target[:, 1])
    return (loss_theta + loss_phi).mean()

# --- [2. 데이터셋: 상세 로드 상태 보고 복구] ---
class FullSequenceDataset(Dataset):
    def __init__(self, pt_dir, csv_path, layer_idx):
        self.pt_dir = pt_dir
        self.layer_key = f"hidden_states{layer_idx}"
        
        df = pd.read_csv(csv_path)
        df = df[df['Status'].astype(str) != '0'].copy()
        
        self.data_list = []
        skipped_no_file = 0
        skipped_parse_error = 0

        for _, row in df.iterrows():
            img_name = str(row['File Name']).strip()
            pt_path = os.path.join(pt_dir, f"{img_name}.pt")
            
            if os.path.exists(pt_path):
                try:
                    dt_str = str(row['DateTime']).strip()
                    dt = datetime.strptime(dt_str, '%Y:%m:%d %H:%M:%S')
                    time_vec = [dt.month, dt.day, dt.hour, dt.minute, dt.second]
                    self.data_list.append({
                        'pt_path': pt_path, 
                        'time_vec': time_vec, 
                        'row_data': row.to_dict()
                    })
                except:
                    skipped_parse_error += 1
            else:
                skipped_no_file += 1
        
        # 데이터 로딩 통계 보고 (복구됨)
        print(f"\n--- [데이터 로딩 요약: Layer {layer_idx}] ---")
        print(f"✅ 유효 데이터: {len(self.data_list)}개")
        print(f"❌ 파일 없음: {skipped_no_file}개")
        print(f"❌ 날짜 파싱 실패: {skipped_parse_error}개")
        print(f"--------------------------------------\n")

    def __len__(self): return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        data = torch.load(item['pt_path'], map_location='cpu')
        hidden_state = data[self.layer_key].to(torch.float32)
        return hidden_state, torch.tensor(item['time_vec'], dtype=torch.float32), item['row_data']

def collate_fn(batch):
    embeddings, times, rows = zip(*batch)
    padded_embeddings = pad_sequence(embeddings, batch_first=True)
    lengths = torch.tensor([e.size(0) for e in embeddings])
    mask = torch.arange(padded_embeddings.size(1))[None, :] < lengths[:, None]
    return padded_embeddings, torch.stack(times), mask, rows

# --- [3. 모델 정의] ---
class SequenceProbeModel(nn.Module):
    def __init__(self, input_dim=4096):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(input_dim, 512), nn.Tanh(), nn.Linear(512, 1))
        self.head = nn.Sequential(nn.Linear(input_dim, 1024), nn.ReLU(), nn.Dropout(0.2), nn.Linear(1024, 2))
    def forward(self, x, mask):
        attn_scores = self.attention(x).squeeze(-1)
        attn_scores = attn_scores.masked_fill(~mask, -1e9)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        context_vec = torch.sum(attn_weights * x, dim=1)
        return self.head(context_vec)

# --- [4. 실행 루프] ---
def run_layer11_full_analysis(pt_dir, csv_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    layer_idx = 11
    
    dataset = FullSequenceDataset(pt_dir, csv_path, layer_idx)
    if len(dataset) < 5: return

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=1, collate_fn=collate_fn)

    model = SequenceProbeModel().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    best_model_state = None
    min_val_loss = float('inf')

    # 학습 단계
    epochs = 20
    for epoch in range(epochs):
        model.train()
        train_pbar = tqdm(train_loader, desc=f"🚀 Layer 11 - Epoch {epoch+1}/{epochs}", leave=False)
        for h_state, t_raw, mask, _ in train_pbar:
            h_state, t_raw, mask = h_state.to(device), t_raw.to(device), mask.to(device)
            target = angular_time_representation(t_raw)
            optimizer.zero_grad()
            loss = cyclic_loss(model(h_state, mask), target)
            loss.backward()
            optimizer.step()
            train_pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        # 매 에포크마다 검증하여 Best Model 저장
        model.eval()
        curr_val_loss = 0
        with torch.no_grad():
            for h_state, t_raw, mask, _ in val_loader:
                h_state, t_raw, mask = h_state.to(device), t_raw.to(device), mask.to(device)
                curr_val_loss += cyclic_loss(model(h_state, mask), angular_time_representation(t_raw)).item()
        
        avg_val_loss = curr_val_loss / len(val_loader)
        if avg_val_loss < min_val_loss:
            min_val_loss = avg_val_loss
            best_model_state = model.state_dict()

    # 최적의 모델로 로드 후 최종 분석
    model.load_state_dict(best_model_state)
    model.eval()
    
    results = []
    errors_sec = []
    
    print("\n🔍 Best Model 기반 최종 분석 및 통계 산출 중...")
    with torch.no_grad():
        for h_state, t_raw, mask, rows in tqdm(val_loader, desc="Inference"):
            h_state, mask = h_state.to(device), mask.to(device)
            pred = model(h_state, mask).cpu().numpy()[0]
            
            row = rows[0]
            actual_dt = datetime.strptime(row['DateTime'], '%Y:%m:%d %H:%M:%S')
            pred_dt = inverse_angular_time(pred[0], pred[1])
            
            # 오차 계산 및 순환 보정
            diff_seconds = abs((actual_dt.replace(year=2026) - pred_dt).total_seconds())
            if diff_seconds > 15768000: diff_seconds = 31536000 - diff_seconds
            errors_sec.append(diff_seconds)
            
            # 홍철 님이 요청하신 리포트 포맷 (복구 및 강화)
            res_entry = row.copy()
            res_entry['Actual_Cycle'] = actual_dt.strftime('%m-%d %H:%M')
            res_entry['Predicted_Cycle'] = pred_dt.strftime('%m-%d %H:%M')
            res_entry['Error_Format'] = str(timedelta(seconds=int(diff_seconds)))
            res_entry['Error_Total_Seconds'] = int(diff_seconds)
            
            # 파싱용 임시 컬럼 삭제
            res_entry.pop('parsed_dt', None)
            results.append(res_entry)

    # 결과 저장
    output_csv = os.path.join(RESULT_SAVE_DIR, "layer11_final_report.csv")
    output_df = pd.DataFrame(results)
    cols = ['File Name', 'DateTime', 'Actual_Cycle', 'Predicted_Cycle', 'Error_Format', 'Description', 'Location', 'Error_Total_Seconds']
    output_df = output_df[[c for c in cols if c in output_df.columns]]
    output_df.to_csv(output_csv, index=False, encoding='utf-8-sig')

    # 통계 리포트 출력 (요청하신 부분)
    if errors_sec:
        print("\n" + "="*50)
        print(f"📊 Layer 11 최종 오차 통계 (연도 무시)")
        print(f"📂 저장 경로: {output_csv}")
        print(f"⏱️ 평균 오차: {timedelta(seconds=int(np.mean(errors_sec)))}")
        print(f"⏱️ 중앙 오차: {timedelta(seconds=int(np.median(errors_sec)))}")
        print(f"⏱️ 최소 오차: {timedelta(seconds=int(np.min(errors_sec)))}")
        print(f"⏱️ 최대 오차: {timedelta(seconds=int(np.max(errors_sec)))}")
        print("="*50)

if __name__ == "__main__":
    PT_DIR = 'runtime/embedding_results_448_pt'
    CSV_PATH = 'runtime/results.csv'
    run_layer11_full_analysis(PT_DIR, CSV_PATH)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# 한글 폰트 설정 (환경에 따라 조정 필요)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def perform_detailed_eda(csv_path):
    # 1. 데이터 로드
    df = pd.read_csv(csv_path)
    
    # 시간 데이터 파싱 (MM-DD HH:MM 형식)
    def parse_cycle(x):
        return datetime.strptime(f"2024-{x}", "%Y-%m-%d %H:%M")

    df['Actual_DT'] = df['Actual_Cycle'].apply(parse_cycle)
    df['Pred_DT'] = df['Predicted_Cycle'].apply(parse_cycle)
    
    # 분석용 속성 추출
    df['Actual_Month'] = df['Actual_DT'].dt.month
    df['Actual_Hour'] = df['Actual_DT'].dt.hour
    df['Error_Hours'] = df['Error_Total_Seconds'] / 3600
    df['Error_Days'] = df['Error_Hours'] / 24

    print("=== 📋 예측 오차 요약 통계 ===")
    stats = df['Error_Total_Seconds'].describe()
    print(f"평균 오차: {timedelta(seconds=int(stats['mean']))}")
    print(f"중앙 오차: {timedelta(seconds=int(stats['50%']))}")
    print(f"최소 오차: {timedelta(seconds=int(stats['min']))}")
    print(f"최대 오차: {timedelta(seconds=int(stats['max']))}")
    print("="*30)

    # 시각화 시작
    fig = plt.figure(figsize=(18, 12))
    
    # (1) 오차 분포 (히스토그램)
    plt.subplot(2, 2, 1)
    sns.histplot(df['Error_Days'], bins=30, kde=True, color='skyblue')
    plt.title('오차 분포 (단위: 일)')
    plt.xlabel('오차 (Days)')
    plt.ylabel('데이터 개수')

    # (2) 실제 시간 vs 예측 시간 산점도
    # (일년을 0~360도 각도로 변환하여 분포 확인)
    plt.subplot(2, 2, 2)
    df['Actual_DayOfYear'] = df['Actual_DT'].dt.dayofyear
    df['Pred_DayOfYear'] = df['Pred_DT'].dt.dayofyear
    plt.scatter(df['Actual_DayOfYear'], df['Pred_DayOfYear'], alpha=0.5, c=df['Error_Days'], cmap='viridis')
    plt.plot([0, 366], [0, 366], 'r--', alpha=0.7) # 정답선
    plt.title('실제 날짜 vs 예측 날짜 (Day of Year)')
    plt.xlabel('실제 날짜 (1~366)')
    plt.ylabel('예측 날짜 (1~366)')
    plt.colorbar(label='오차 (Days)')

    # (3) 월별 평균 오차 (계절별 성능)
    plt.subplot(2, 2, 3)
    sns.barplot(data=df, x='Actual_Month', y='Error_Days', palette='coolwarm')
    plt.title('월별 평균 예측 오차')
    plt.xlabel('월 (Month)')
    plt.ylabel('평균 오차 (Days)')

    # (4) 시간대별 평균 오차 (Day/Night 성능)
    plt.subplot(2, 2, 4)
    sns.lineplot(data=df, x='Actual_Hour', y='Error_Hours', marker='o', color='orange')
    plt.xticks(range(0, 24))
    plt.grid(True, alpha=0.3)
    plt.title('시간대별 평균 예측 오차')
    plt.xlabel('시간 (Hour)')
    plt.ylabel('평균 오차 (Hours)')

    plt.tight_layout()
    plt.savefig('prediction_analysis_plots.png')
    plt.show()

    # 2. 오차 원인 분석을 위한 상위/하위 샘플 추출
    print("\n=== 🚀 가장 잘 맞춘 예측 (Top 5) ===")
    print(df.sort_values('Error_Total_Seconds').head(5)[['File Name', 'Actual_Cycle', 'Predicted_Cycle', 'Error_Format']])

    print("\n=== 🛑 오차가 가장 큰 예측 (Worst 5) ===")
    print(df.sort_values('Error_Total_Seconds', ascending=False).head(5)[['File Name', 'Actual_Cycle', 'Predicted_Cycle', 'Error_Format']])

# 실행
if __name__ == "__main__":
    path = 'runtime/layer11_final_report.csv'
    perform_detailed_eda(path)

In [ ]:
import os
import json
import re
import gzip
import pandas as pd
from tqdm import tqdm

# ===============================
# 경로 설정
# ===============================
COCO_JSON_PATH = 'runtime/captions_train2017.json'
YFCC_META_DIR = 'runtime/yfcc_meta'
SAVE_PATH = 'runtime/coco_yfcc_geo_time.csv'

# ===============================
# [1] COCO Flickr photo_id 추출
# ===============================
print("COCO JSON 분석 중...")

with open(COCO_JSON_PATH, "r", encoding="utf-8") as f:
    coco = json.load(f)

coco_id_map = {}

for img in coco["images"]:
    url = img.get("flickr_url", "")
    if not url:
        continue

    try:
        # 예:
        # http://farm9.staticflickr.com/8186/8119368305_4e622c8349_z.jpg
        # 실제 photo_id = 8119368305

        filename = url.split("/")[-1]          # 8119368305_4e622c8349_z.jpg
        photo_id = filename.split("_")[0]     # 8119368305

        if photo_id.isdigit():
            coco_id_map[photo_id] = {
                "coco_id": img["id"],
                "file_name": img["file_name"]
            }

    except Exception:
        continue

target_ids = set(coco_id_map.keys())

print(f"검색 대상 COCO ID 수: {len(target_ids):,}")

# 샘플 확인
print("샘플 photo_id 10개:")
print(list(target_ids)[:10])

# ===============================
# [2] YFCC 전체 스캔
# ===============================
results = []
processed_lines = 0
error_count = 0

shard_files = [os.path.join(YFCC_META_DIR, f"{i:03x}.gz") for i in range(4096)]
shard_files = [f for f in shard_files if os.path.exists(f)]

print(f"실제 존재 확인된 shard 파일 수: {len(shard_files)}")

pbar = tqdm(shard_files, desc="YFCC 정밀 스캔 중")

for shard_path in pbar:
    with gzip.open(shard_path, "rt", encoding="utf-8", errors="ignore") as f:
        for line in f:
            processed_lines += 1
            line = line.strip()

            if not line:
                continue

            try:
                start_idx = line.find("{")
                end_idx = line.rfind("}")

                if start_idx == -1 or end_idx == -1:
                    continue

                json_str = line[start_idx:end_idx + 1]
                data = json.loads(json_str)

                yfcc_pid = str(data.get("photoid", "")).strip()

                if yfcc_pid in target_ids:
                    meta = coco_id_map[yfcc_pid]

                    results.append({
                        "coco_id": meta["coco_id"],
                        "file_name": meta["file_name"],
                        "photo_id": yfcc_pid,
                        "latitude": data.get("latitude"),
                        "longitude": data.get("longitude"),
                        "taken_time": data.get("datetaken")
                    })

                    if len(results) % 10 == 0:
                        pbar.set_postfix({
                            "Matched": len(results),
                            "Lines": f"{processed_lines:,}",
                            "Err": error_count
                        })

            except Exception:
                error_count += 1
                continue

    pbar.set_postfix({
        "Matched": len(results),
        "Lines": f"{processed_lines:,}",
        "Err": error_count
    })

# ===============================
# [3] 저장
# ===============================
print("\n" + "=" * 40)
print("최종 스캔 결과")
print(f"총 읽은 라인 수: {processed_lines:,}")
print(f"최종 매칭 성공 수: {len(results):,}")
print(f"에러 수: {error_count:,}")
print("=" * 40)

print("\n" + "=" * 40)
print("최종 스캔 결과")
print(f"총 읽은 라인 수: {processed_lines:,}")
print(f"최종 매칭 성공 수: {len(results):,}")
print(f"에러 수: {error_count:,}")
print("=" * 40)

if results:
    # 1. 리스트를 데이터프레임으로 변환
    result_df = pd.DataFrame(results)
    
    # 2. 파일명(file_name)을 기준으로 중복 제거 (일관성 유지의 핵심)
    # COCO 데이터셋의 물리적 이미지 파일명과 1:1 매칭되도록 합니다.
    result_df = result_df.drop_duplicates(subset=["file_name"])
    
    # 3. 컬럼 순서 재배치 (파일명이 가장 먼저 보이게)
    cols = ["file_name", "coco_id", "photo_id", "latitude", "longitude", "taken_time"]
    result_df = result_df[cols]
    
    # 4. 저장
    SAVE_PATH = 'runtime/coco_yfcc_geo_time.csv'
    result_df.to_csv(SAVE_PATH, index=False, encoding="utf-8-sig")
    
    print(f"결과 저장 완료: {SAVE_PATH}")
    print(f"고유 이미지 파일명 수: {len(result_df):,}")
else:
    print("매칭 결과 없음")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
import os

# ------------------------------------------------------------
# 1. 데이터 로드 및 전처리
# ------------------------------------------------------------
CSV_PATH = 'runtime/coco_yfcc_geo_time.csv'
OUTPUT_DIR = 'runtime/eda_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 데이터 로드
df = pd.read_csv(CSV_PATH)

# [시간 데이터 처리]
df['dt'] = pd.to_datetime(df['taken_time'], errors='coerce')
df = df.dropna(subset=['dt']) # 날짜가 없는 데이터는 분석에서 제외

# 분석용 파생 변수 생성
df['year'] = df['dt'].dt.year.astype(int)
df['month'] = df['dt'].dt.month
df['hour'] = df['dt'].dt.hour
df['dayofweek'] = df['dt'].dt.day_name()

# [연도 데이터 필터링] 
# 데이터의 메타데이터 오류(예: 1700년 등)를 방지하기 위해 2000년~2025년 사이 데이터만 유지
df = df[(df['year'] >= 2000) & (df['year'] <= 2025)]

# [카테고리 순서 지정] 요일과 월이 섞이지 않도록 함
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['dayofweek'] = pd.Categorical(df['dayofweek'], categories=day_order, ordered=True)

# [지리 데이터 처리]
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# ------------------------------------------------------------
# 2. 시각화 (Time-Series Analysis)
# ------------------------------------------------------------
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# (1) 연도별 이미지 수 (Trend) - X축 가독성 개선
year_counts = df['year'].value_counts().sort_index()
sns.barplot(x=year_counts.index, y=year_counts.values, ax=axes[0, 0], palette="viridis")
axes[0, 0].set_title('Images per Year (Filtered 2000-2025)', fontsize=15)
axes[0, 0].tick_params(axis='x', rotation=45)

# 데이터가 너무 많을 경우 X축 라벨을 2개당 하나씩만 표시하여 겹침 방지
for i, label in enumerate(axes[0, 0].get_xticklabels()):
    if i % 2 != 0: # 홀수 번째 라벨은 숨김 (필요시 조절)
        label.set_visible(True)

# (2) 월별 이미지 수 (Seasonality)
sns.countplot(data=df, x='month', ax=axes[0, 1], palette="magma")
axes[0, 1].set_title('Images per Month (Seasonality)', fontsize=15)
axes[0, 1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])

# (3) 시간대별 촬영 분포 (Daily Pattern)
sns.histplot(df['hour'], bins=24, kde=True, color='teal', ax=axes[1, 0])
axes[1, 0].set_title('Distribution of Taken Time (Hour of Day)', fontsize=15)
axes[1, 0].set_xlabel('Hour (0-23)')

# (4) 요일별 이미지 수 (Weekly Pattern)
sns.countplot(data=df, x='dayofweek', ax=axes[1, 1], palette="coolwarm")
axes[1, 1].set_title('Images per Day of Week', fontsize=15)

plt.tight_layout()
report_path = os.path.join(OUTPUT_DIR, 'time_eda_report_fixed.png')
plt.savefig(report_path)
print(f"✅ 차트 리포트 저장 완료: {report_path}")

# ------------------------------------------------------------
# 3. 지리 정보 시각화 (Heatmap)
# ------------------------------------------------------------
geo_data = df.dropna(subset=['latitude', 'longitude'])

if not geo_data.empty:
    # 지도의 중심점 계산
    center_lat = geo_data['latitude'].mean()
    center_lon = geo_data['longitude'].mean()
    
    m = folium.Map(location=[center_lat, center_lon], zoom_start=2)
    
    # 데이터가 너무 많을 경우 성능을 위해 샘플링 (옵션: .sample(10000))
    heat_data = geo_data[['latitude', 'longitude']].values.tolist()
    HeatMap(heat_data, radius=8, blur=12).add_to(m)
    
    map_path = os.path.join(OUTPUT_DIR, 'world_heatmap.html')
    m.save(map_path)
    print(f"✅ 지오 히트맵 저장 완료: {map_path} ({len(geo_data)} points)")
else:
    print("❌ 위도/경도 데이터가 없어 지도를 생성하지 못했습니다.")

In [ ]:
#이것은 YFCC 메타데이터에서 COCO 이미지와 매칭된 사진들의 촬영 시간과 위치 정보를 분석하기 위한 코드입니다.

import pandas as pd
import numpy as np

# 1. 파일 경로 설정
base_file_path = 'runtime/coco_sharegpt_integrated.csv'
geo_time_file_path = 'runtime/coco_yfcc_geo_time.csv'
output_file_path = 'runtime/coco_sharegpt_integrated_yfcc.csv'

# 2. 데이터 불러오기
df_base = pd.read_csv(base_file_path)
df_geo = pd.read_csv(geo_time_file_path)

# 3. 데이터 병합 전 처리
# 베이스 파일에서 덮어씌울 대상인 기존 DateTime, Location, Status 컬럼이 있다면 제거하여 충돌을 방지합니다.
# (File Name, Description, Full Path만 남기고 나머지는 새로 결합할 데이터로 채웁니다)
df_base_cleaned = df_base[['File Name', 'Description', 'Full Path']]

# 4. 데이터 병합 (File Name 기준 Left Join)
df_merged = pd.merge(
    df_base_cleaned, 
    df_geo[['file_name', 'latitude', 'longitude', 'taken_time']], 
    left_on='File Name', 
    right_on='file_name', 
    how='left'
)

# 5. 데이터 가공 및 덮어쓰기 로직
# Location: (latitude,longitude) 형태로 새로 생성
def format_location(row):
    if pd.notnull(row['latitude']) and pd.notnull(row['longitude']):
        return f"({row['latitude']},{row['longitude']})"
    return np.nan

df_merged['Location'] = df_merged.apply(format_location, axis=1)

# DateTime: taken_time 값으로 덮어씌움
df_merged['DateTime'] = df_merged['taken_time']

# 6. Status 값 결정 (새 기준 적용)
# Location과 DateTime 두 개 다 있으면 2, DateTime만 있으면 1, 그 외 0
def set_status(row):
    has_loc = pd.notnull(row['Location'])
    has_time = pd.notnull(row['DateTime'])
    
    if has_loc and has_time:
        return 2
    elif has_time:
        return 1
    else:
        return 0

df_merged['Status'] = df_merged.apply(set_status, axis=1)

# 7. 최종 컬럼 정리 및 저장
# 요청하신 순서: File Name, Description, DateTime, Location, Full Path, Status
final_columns = ['File Name', 'Description', 'DateTime', 'Location', 'Full Path', 'Status']
df_final = df_merged[final_columns]

# 결과 저장
df_final.to_csv(output_file_path, index=False, encoding='utf-8-sig')

print(f"기존 데이터를 덮어씌워 저장을 완료했습니다: {output_file_path}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
import os
import pandas as pd
import math
from datetime import datetime
from tqdm import tqdm

# --- [1. 로깅 및 결과 저장 설정] ---
RESULT_DIR = "qformer_analysis_logs"
os.makedirs(RESULT_DIR, exist_ok=True)

# --- [2. 시간 표현형 및 손실 함수] ---
def angular_time_representation_refined(T):
    month, day, hour, minute, _ = torch.chunk(T.float(), 5, dim=1)
    month_d = (month - 1) + (day - 1) * (1 / 30.44)
    theta = 2 * math.pi * month_d / 12 - math.pi
    hour_d = hour + minute * (1 / 60.0)
    phi = 2 * math.pi * hour_d / 24 - math.pi
    return torch.cat((theta, phi), dim=1)

def weighted_cyclic_loss(pred, target, time_weight=3.0):
    loss_theta = 1 - torch.cos(pred[:, 0] - target[:, 0])
    loss_phi = 1 - torch.cos(pred[:, 1] - target[:, 1])
    return (loss_theta + (loss_phi * time_weight)).mean()

class QFormerDataset(Dataset):

    def __init__(self, pt_dir, csv_path, target_layers=[11, 24, 19]):

        self.pt_dir = pt_dir

        self.target_layers = target_layers

        df = pd.read_csv(csv_path)

        df = df[df['Status'].astype(str) != '0'].copy()

       

        self.data_list = []

        for _, row in df.iterrows():

            pt_path = os.path.join(pt_dir, f"{row['File Name']}.pt")

            if os.path.exists(pt_path):

                try:

                    dt = datetime.strptime(str(row['DateTime']), '%Y:%m:%d %H:%M:%S')

                    time_vec = [dt.month, dt.day, dt.hour, dt.minute, dt.second]

                    self.data_list.append((pt_path, time_vec))

                except: continue

        print(f"✅ 데이터 {len(self.data_list)}개 로드 완료 (Layers: {target_layers})")



    def __len__(self): return len(self.data_list)



    def __getitem__(self, idx):

        pt_path, time_vec = self.data_list[idx]

        data = torch.load(pt_path, map_location='cpu')

        # 3개 레이어를 시퀀스 방향으로 이어붙임 (Long Sequence 생성)

        # [Seq*3, 4096] 형태가 됨

        combined_seq = torch.cat([data[f"hidden_states{i}"].to(torch.float32) for i in self.target_layers], dim=0)

        return combined_seq, torch.tensor(time_vec, dtype=torch.float32)



def qformer_collate(batch):

    sequences, times = zip(*batch)

    padded_seqs = pad_sequence(sequences, batch_first=True)

    lengths = torch.tensor([s.size(0) for s in sequences])

    mask = torch.arange(padded_seqs.size(1))[None, :] < lengths[:, None]

    return padded_seqs, torch.stack(times), mask



# --- [3. 정통 Q-Former 모델 (4096-dim)] ---
class AuthenticQFormer(nn.Module):
    def __init__(self, input_dim=4096, num_queries=8, num_heads=8): # 쿼리 32 -> 8 축소
        super().__init__()
        self.num_queries = num_queries
        self.queries = nn.Parameter(torch.randn(1, num_queries, input_dim))
        
        # 가중치 초기화 (Xavier/Kaiming)를 통해 초기 안정성 확보
        nn.init.xavier_uniform_(self.queries)

        self.self_attn = nn.MultiheadAttention(input_dim, num_heads, batch_first=True, dropout=0.1)
        self.cross_attn = nn.MultiheadAttention(input_dim, num_heads, batch_first=True, dropout=0.1)
        
        self.norm1 = nn.LayerNorm(input_dim)
        self.norm2 = nn.LayerNorm(input_dim)
        self.norm3 = nn.LayerNorm(input_dim)
        
        self.ffn = nn.Sequential(
            nn.Linear(input_dim, 1024), # FFN 크기도 살짝 줄임 (2048 -> 1024)
            nn.GELU(),
            nn.Dropout(0.5), # 드롭아웃 강화
            nn.Linear(1024, input_dim)
        )
        
        self.output_head = nn.Sequential(
            nn.Linear(input_dim * num_queries, 512), # 병목 구간 생성
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.5), # 드롭아웃 강화
            nn.Linear(512, 2)
        )

    def forward(self, image_embeds, mask):
        # [훈련 시에만] 입력 임베딩에 미세한 노이즈 추가 (Overfitting 방지 핵심)
        if self.training:
            noise = torch.randn_like(image_embeds) * 0.01 # 1% 정도의 노이즈
            image_embeds = image_embeds + noise

        b = image_embeds.size(0)
        query_embeds = self.queries.expand(b, -1, -1)
        
        # Self-Attention: 쿼리 간 정보 교환
        query_embeds = query_embeds + self.self_attn(query_embeds, query_embeds, query_embeds)[0]
        query_embeds = self.norm1(query_embeds)
        
        # Cross-Attention: 쿼리가 이미지 시퀀스에서 시간 정보를 추출
        attn_out, _ = self.cross_attn(query=query_embeds, 
                                     key=image_embeds, 
                                     value=image_embeds, 
                                     key_padding_mask=~mask)
        query_embeds = query_embeds + attn_out
        query_embeds = self.norm2(query_embeds)
        
        query_embeds = query_embeds + self.ffn(query_embeds)
        query_embeds = self.norm3(query_embeds)
        
        out = query_embeds.view(b, -1)
        return self.output_head(out)

# --- [4. 학습 및 세부 로깅 함수] ---
def train_qformer(pt_dir, csv_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 학습 시작 | 디바이스: {device}")

    # 데이터 로딩
    dataset = QFormerDataset(pt_dir, csv_path)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=qformer_collate)
    val_loader = DataLoader(val_ds, batch_size=4, collate_fn=qformer_collate)

    model = AuthenticQFormer().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    # 에러 수정: 루프 시작 전 변수 초기화
    best_loss = float('inf')
    counter = 0
    patience = 12 # 모델이 무거우므로 인내심을 조금 더 늘렸습니다.

    for epoch in range(100):
        # --- [TRAIN PHASE] ---
        model.train()
        train_loss = 0
        train_pbar = tqdm(train_loader, desc=f"🌙 Epoch {epoch:02d} [Train]", leave=False)
        
        for seqs, times, mask in train_pbar:
            seqs, mask = seqs.to(device), mask.to(device)
            target = angular_time_representation_refined(times.to(device))
            
            optimizer.zero_grad()
            pred = model(seqs, mask)
            loss = weighted_cyclic_loss(pred, target, time_weight=2.5)
            loss.backward()
            
            # Gradient Clipping (4096 차원 폭주 방지)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.5)
            
            optimizer.step()
            train_loss += loss.item()
            train_pbar.set_postfix({"batch_loss": f"{loss.item():.4f}"})
        
        avg_train_loss = train_loss / len(train_loader)
        
        # --- [VAL PHASE] ---
        model.eval()
        val_loss = 0
        val_pbar = tqdm(val_loader, desc=f"⭐ Epoch {epoch:02d} [Val]", leave=False)
        
        with torch.no_grad():
            for seqs, times, mask in val_pbar:
                seqs, mask = seqs.to(device), mask.to(device)
                pred = model(seqs, mask)
                v_loss = weighted_cyclic_loss(pred, angular_time_representation_refined(times.to(device)), 3.0)
                val_loss += v_loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        scheduler.step()

        # --- [LOGGING] ---
        current_lr = optimizer.param_groups[0]['lr']
        print(f"📈 [Epoch {epoch:02d}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.6f} | LR: {current_lr:.2e}")

        # Best Model 저장 및 Early Stopping 체크
        if avg_val_loss < best_loss:
            print(f"✨ New Best! ({best_loss:.4f} -> {avg_val_loss:.4f}) - 모델 저장됨")
            best_loss = avg_val_loss
            torch.save(model.state_dict(), "best_qformer_model.pth")
            counter = 0
        else:
            counter += 1
            print(f"⚠️ 개선 실패 ({counter}/{patience})")
            if counter >= patience:
                print("🛑 Early Stopping: 학습을 종료합니다.")
                break

    print(f"\n✅ 최종 학습 완료! 최저 손실값: {best_loss:.6f}")

if __name__ == "__main__":

    PT_DIR = 'runtime/embedding_results_448_pt'

    CSV_PATH = 'runtime/results.csv'
    train_qformer(PT_DIR, CSV_PATH)

In [ ]:
import torch
import math
from datetime import datetime, timedelta

# --- [1. 복구 로직: 각도(Rad) -> 실제 시간] ---
def decode_prediction(theta, phi):
    """
    theta: [-pi, pi] -> Month, Day
    phi: [-pi, pi] -> Hour, Minute, Second
    """
    # 1. 월/일 복구 (365일 기준)
    # 0~2pi 범위로 변환 후 12개월로 스케일링
    month_total_days = ((theta + math.pi) / (2 * math.pi)) * 365
    
    # 가상의 시작일(1월 1일)에서 offset만큼 더함
    start_of_year = datetime(2026, 1, 1) # 기준 연도
    predicted_date = start_of_year + timedelta(days=float(month_total_days))
    
    # 2. 시/분/초 복구 (24시간 기준)
    hour_total_seconds = ((phi + math.pi) / (2 * math.pi)) * 24 * 3600
    predicted_time = timedelta(seconds=float(hour_total_seconds))
    
    # 최종 결과 조합
    final_dt = predicted_date.replace(hour=0, minute=0, second=0) + predicted_time
    return final_dt

# --- [2. 추론 함수] ---
def predict_time_from_layers(layer_data_list, model_path="best_ensemble_model.pth"):
    """
    layer_data_list: [L11, L24, L19, L9, L13] 각 [Seq, 4096] 형태의 텐서 리스트
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 모델 초기화 및 로드
    model = EnsembleTimeProbe(num_layers=5).to(device)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    
    # 데이터 준비 (Batch=1 처리)
    # layers: [[1, Seq, 4096], ...]
    layers = [l.unsqueeze(0).to(device) for l in layer_data_list]
    
    # 마스크 생성 (패딩 없음, 모두 True)
    mask = torch.ones((1, layers[0].size(1)), dtype=torch.bool).to(device)
    
    with torch.no_grad():
        pred = model(layers, mask) # [1, 2] -> [theta, phi]
        theta, phi = pred[0, 0].item(), pred[0, 1].item()
        
    return decode_prediction(theta, phi)

# --- [3. 실행 예시] ---
# 홍철 님이 새로 뽑은 사진의 .pt 파일을 로드했다고 가정합니다.
sample_pt = torch.load('runtime/20251220_145915.jpg.pt')
target_layers = [11, 24, 19, 9, 13]
inputs = [sample_pt[f"hidden_states{i}"].float() for i in target_layers]

result = predict_time_from_layers(inputs)
print(f"🚀 모델의 예측 결과: {result.strftime('%Y년 %m월 %d일 %H시 %M분 %S초')}")

In [ ]:
import torch

# 파일 경로
file_path = 'runtime/1345622112690.jpg.pt'

# 데이터 로드
try:
    data = torch.load(file_path, map_location='cpu')
    
    # 특정 레이어(9번) 데이터 확인
    # 이전 코드에서 키값이 "hidden_states9"로 저장되었으므로 해당 키를 사용합니다.
    target_key = "hidden_states9"
    
    if target_key in data:
        embedding = data[target_key]
        print(f"✅ 필드명: {target_key}")
        print(f"📊 데이터 형태 (Shape): {embedding.shape}")
        print(f"🔢 차원 수: {embedding.ndim}차원")
        print(f"🧪 데이터 타입 (Dtype): {embedding.dtype}")
    else:
        print(f"❌ '{target_key}' 키를 찾을 수 없습니다. 존재하는 키: {list(data.keys())}")

except Exception as e:
    print(f"에러 발생: {e}")